# SRQ-FLY — locked three-dataset held-out study
Near-final reporting notebook for CIFAR-100, CUB-200-2011 and the disclosed legacy ImageNet-R processed split. It verifies the completed train-only selections, extracts frozen training features, locks an immutable authorization, then runs six paired held-out seeds without test tuning or accuracy-based early stopping.

In [ ]:
# === Edit repository/path values only. Do not edit seeds or method settings. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'paper/srq-fly-draft'
WORK_DIR = '/content/SOHO-CL'
ARTIFACT_DIR = '/content/srq_train_only_evidence'
DATA_ROOT = '/content/srq_datasets'
FEATURE_CACHE_ROOT = '/content/srq_final_feature_caches'
WTA_CACHE_ROOT = '/content/srq_final_wta_caches'
OUTPUT_ROOT = '/content/srq_final_outputs'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_MANIFEST_SHA256 = '58036d4c282293eeca694d8e5f895bc260b4f8d6f13059c80cd60fac0aa2bd72'
EXPECTED_RUNNER_SHA256 = '4ef35968912f459579607c3eb4fbddd5d855a231fe6f34ef1ec9c1b2e9985d12'
EXPECTED_EXTRACTOR_SHA256 = '09f97bfd9f96fc4f8dd93d00d92318fe307050db545debef02e3c54697c404c1'

In [ ]:
# Fresh clone, dependencies, GPU and immutable source checks.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
MANIFEST = 'configs/srq_fly_three_dataset_heldout.json'
assert sha(MANIFEST) == EXPECTED_MANIFEST_SHA256
assert sha('tools/srq_fly_heldout.py') == EXPECTED_RUNNER_SHA256
assert sha('tools/srq_fly_extract_test.py') == EXPECTED_EXTRACTOR_SHA256
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Repository must be clean.'
print('GPU:', torch.cuda.get_device_name(0))
print('repo commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('manifest SHA-256:', EXPECTED_MANIFEST_SHA256)

In [ ]:
# Download the exact frozen ViT checkpoint and the three processed datasets.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
DATASET_ROOTS = {
  'cifar100': kagglehub.dataset_download('zaphat206/cifar-100'),
  'cub200': kagglehub.dataset_download('zaphat206/cub-200-2011'),
  'imagenetr': kagglehub.dataset_download('zaphat206/imagenet-r'),
}
print('checkpoint:', CHECKPOINT_PATH)
print(json.dumps(DATASET_ROOTS, indent=2))

In [ ]:
# Feature-free dataset identity audit. ImageNet-R exit code 2 is the locked overlap disclosure.
CUB_AUDIT = '/content/cub_final_audit.json'
IMAGENETR_AUDIT = '/content/imagenetr_final_audit.json'
cub = subprocess.run([sys.executable,'-u','tools/cub_dataset_audit.py','--root',DATASET_ROOTS['cub200'],'--output',CUB_AUDIT,'--expected-identity-sha256','e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca'])
assert cub.returncode == 0
imagenetr = subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOTS['imagenetr'],'--output',IMAGENETR_AUDIT,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4'])
assert imagenetr.returncode == 2, 'Expected the preregistered legacy-overlap status.'
audit = json.loads(Path(IMAGENETR_AUDIT).read_text())
assert audit['cross_split_duplicate_content_count'] == 19
assert audit['cross_split_conflicting_label_duplicate_count'] == 18
print('DATASET AUDIT PASS; ImageNet-R remains a disclosed legacy processed split.')

In [ ]:
# TRAIN-ONLY GATE: upload and verify the four completed selection artifacts. No test is opened.
from google.colab import files
artifact_dir = Path(ARTIFACT_DIR); artifact_dir.mkdir(parents=True, exist_ok=True)
required = {
 'srq_fly_cifar_d5_train_only.zip',
 'srq_fly_cub_d4_multiseed_train_only.zip',
 'srq_fly_imagenetr_d21_lambda_robustness.zip',
 'srq_fly_imagenetr_d1_train_only.zip',
}
missing = sorted(name for name in required if not (artifact_dir/name).is_file())
if missing:
    print('Upload these train-only ZIPs:', missing)
    uploaded = files.upload()
    for name, content in uploaded.items(): (artifact_dir/name).write_bytes(content)
assert all((artifact_dir/name).is_file() for name in required), 'Missing train-only evidence.'
subprocess.run([sys.executable,'-u','tools/srq_fly_heldout.py','verify-selection','--manifest',MANIFEST,'--artifact-dir',ARTIFACT_DIR], check=True)
print('TRAIN-ONLY HYPERPARAMETER GATE: PASS — settings are now immutable.')

In [ ]:
# Extract TRAIN features only. Each task prints one progress line; test.pt must remain absent.
manifest = json.loads(Path(MANIFEST).read_text())
Path(FEATURE_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
for key in ('cifar100','cub200','imagenetr'):
    cfg = manifest['datasets'][key]; cache = Path(FEATURE_CACHE_ROOT)/key
    if not (cache/'train.pt').is_file():
        command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only',
          '--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,
          '--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b',
          '--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_{key}',
          '--dataset',cfg['dataset'],'--model-name','vit_base_patch16_224','--data-augmentation','vit',
          '--seed','2025','--num-classes',str(cfg['num_classes']),'--num-tasks',str(cfg['num_tasks']),
          '--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
        print(f'TRAIN EXTRACT START {key}', flush=True); subprocess.run(command, check=True)
    assert (cache/'train.pt').is_file()
    if (cache/'test.pt').is_file():
        assert Path(OUTPUT_ROOT,'heldout_authorization.json').is_file(), 'test.pt appeared before authorization'
    else: print(f'TRAIN CACHE READY {key}; test.pt absent')
print('ALL TRAIN-ONLY FEATURE CACHES READY')

In [ ]:
# Synthetic correctness and leakage tests only.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_srq_fly_heldout.py','tests/test_srq_fly_math.py','tests/test_srq_fly_learner.py'], check=True)
print('FINAL RUNNER CORRECTNESS GATE: PASS')

## Single-use boundary
Running the next cell freezes the manifest and records authorization before any held-out feature is extracted. From that point onward, do not edit method settings based on test output. Infrastructure failures may resume only with the identical commit and manifest.

In [ ]:
# Lock single-use authorization. Repository must still be clean.
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'-u','tools/srq_fly_heldout.py','authorize','--manifest',MANIFEST,'--artifact-dir',ARTIFACT_DIR,'--output-root',OUTPUT_ROOT,'--require-clean-git'], check=True)
AUTHORIZATION = str(Path(OUTPUT_ROOT)/'heldout_authorization.json')
print(json.dumps(json.loads(Path(AUTHORIZATION).read_text()), indent=2))

In [ ]:
# Extract HELD-OUT TEST features once, after authorization. Progress is one line per task.
for key in ('cifar100','cub200','imagenetr'):
    command = [sys.executable,'-u','tools/srq_fly_extract_test.py','--manifest',MANIFEST,'--dataset-key',key,
      '--authorization',AUTHORIZATION,'--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),
      '--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,'--device','cuda',
      '--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print(f'HELDOUT EXTRACTION START {key}', flush=True); subprocess.run(command, check=True)
print('ALL HELD-OUT FEATURE CACHES READY')

In [ ]:
# Helper: run/resume every locked seed and print a compact dataset table.
import pandas as pd
def run_dataset(key, audit=None):
    command = [sys.executable,'-u','tools/srq_fly_heldout.py','evaluate','--manifest',MANIFEST,'--dataset-key',key,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--code-cache-root',WTA_CACHE_ROOT,
      '--output-root',OUTPUT_ROOT,'--authorization',AUTHORIZATION,'--device','cuda']
    if audit: command += ['--dataset-audit',audit]
    print(f'LOCKED HELDOUT START {key}: six seeds × four methods', flush=True)
    subprocess.run(command, check=True)
    payload=json.loads(Path(OUTPUT_ROOT,key,'heldout_results.json').read_text())
    rows=[]
    for seed_result in payload['seed_results']:
      for method,result in seed_result['methods'].items():
        rows.append({'seed':seed_result['seed'],'method':method,'status':result['status'],
          'final_accuracy':result.get('final_accuracy'),'average_incremental_accuracy':result.get('average_incremental_accuracy'),
          'forgetting':result.get('forgetting'),'state_bytes':result.get('persistent_state_bytes')})
    display(pd.DataFrame(rows)); print('DATASET STATUS:', payload['status'])


In [ ]:
# CIFAR-100 held-out run. Safe to rerun after interruption; completed units restore from disk.
run_dataset('cifar100')

In [ ]:
# CUB-200-2011 held-out run.
run_dataset('cub200', CUB_AUDIT)

In [ ]:
# ImageNet-R legacy processed-split run; every report retains the overlap disclosure.
run_dataset('imagenetr', IMAGENETR_AUDIT)

In [ ]:
# Final mean ± std/CI tables and compact evidence ZIP. Feature/WTA caches are deliberately excluded.
subprocess.run([sys.executable,'-u','tools/srq_fly_heldout.py','summarize','--manifest',MANIFEST,'--output-root',OUTPUT_ROOT], check=True)
table = pd.read_csv(Path(OUTPUT_ROOT)/'three_dataset_summary.csv')
display(table.sort_values(['dataset','method']))
summary = json.loads(Path(OUTPUT_ROOT,'three_dataset_summary.json').read_text())
print('Paired SRQ minus state-matched FLY:', json.dumps(summary['paired_srq_minus_state_matched_fly'], indent=2))
print('ImageNet-R disclosure:', summary['imagenetr_disclosure'])
archive = '/content/srq_fly_three_dataset_heldout_results.zip'
shutil.make_archive(archive[:-4], 'zip', OUTPUT_ROOT)
print('ZIP bytes:', Path(archive).stat().st_size, 'SHA-256:', sha(archive))
files.download(archive)